# Notebook 1 — ETL Pipeline Olist + Brazil Holiday API

Notebook ini adalah versi perbaikan untuk **Pipeline ETL** Tugas Besar Big Data.

Perbaikan utama versi ini:
- Cell Data Understanding sudah aman untuk kolom API yang berbentuk `list`.
- Join sudah menggunakan **controlled join** dengan grain `order_id + order_item_id`.
- `payments` dan `reviews` diagregasi sebelum join untuk mencegah duplikasi tidak valid.
- Fact table final memiliki lebih dari 12 kolom.
- Query SQL analitik ditambah menjadi **8 query**.
- File `schema.sql` dibuat untuk dokumentasi PK/FK star schema.


## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Import Library dan Setup Path

In [ ]:
import os
import time
import json
import sqlite3
import requests
import numpy as np
import pandas as pd
from datetime import datetime

PROJECT_PATH = '/content/drive/MyDrive/bigdata_final_project'
RAW_OLIST_PATH = f'{PROJECT_PATH}/raw/olist'
RAW_HOLIDAY_PATH = f'{PROJECT_PATH}/raw/holidays'
DATALAKE_RAW_PATH = f'{PROJECT_PATH}/datalake/raw_zone'
DATALAKE_PROCESSED_PATH = f'{PROJECT_PATH}/datalake/processed_zone'
DATALAKE_CURATED_PATH = f'{PROJECT_PATH}/datalake/curated_zone'
WAREHOUSE_PATH = f'{PROJECT_PATH}/warehouse'
LOG_PATH = f'{PROJECT_PATH}/etl_pipeline/logs'

for path in [RAW_HOLIDAY_PATH, DATALAKE_RAW_PATH, DATALAKE_PROCESSED_PATH, DATALAKE_CURATED_PATH, WAREHOUSE_PATH, LOG_PATH]:
    os.makedirs(path, exist_ok=True)

print('Project path siap:', PROJECT_PATH)

Project path siap: /content/drive/MyDrive/bigdata_final_project


## 2. Fungsi Logging ETL

In [ ]:
etl_logs = []

def add_log(step, source_name, rows=None, cols=None, file_size_mb=None, execution_time_sec=None, status='success', notes=''):
    etl_logs.append({
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'step': step,
        'source_name': source_name,
        'rows': rows,
        'cols': cols,
        'file_size_mb': file_size_mb,
        'execution_time_sec': execution_time_sec,
        'status': status,
        'notes': notes
    })

def save_logs():
    log_df = pd.DataFrame(etl_logs)
    log_file = f'{LOG_PATH}/etl_process_log.csv'
    log_df.to_csv(log_file, index=False)
    print('Log ETL tersimpan di:', log_file)
    return log_df

def get_file_size_mb(file_path):
    if os.path.exists(file_path):
        return round(os.path.getsize(file_path) / (1024 * 1024), 3)
    return None

## 3. Extract Source 1 — Olist CSV dari Kaggle

In [ ]:
def extract_etl_source1():
    start_time = time.time()
    source_name = 'Kaggle Olist CSV Dataset'

    file_map = {
        'customers': 'olist_customers_dataset.csv',
        'geolocation': 'olist_geolocation_dataset.csv',
        'order_items': 'olist_order_items_dataset.csv',
        'payments': 'olist_order_payments_dataset.csv',
        'reviews': 'olist_order_reviews_dataset.csv',
        'orders': 'olist_orders_dataset.csv',
        'products': 'olist_products_dataset.csv',
        'sellers': 'olist_sellers_dataset.csv',
        'translation': 'product_category_name_translation.csv'
    }

    data = {}
    for table_name, file_name in file_map.items():
        file_path = f'{RAW_OLIST_PATH}/{file_name}'
        df = pd.read_csv(file_path)
        data[table_name] = df

        raw_copy_path = f'{DATALAKE_RAW_PATH}/{file_name}'
        df.to_csv(raw_copy_path, index=False)

        add_log(
            step='extract',
            source_name=f'{source_name} - {table_name}',
            rows=df.shape[0],
            cols=df.shape[1],
            file_size_mb=get_file_size_mb(file_path),
            notes=f'Raw file copied to {raw_copy_path}'
        )

    execution_time = round(time.time() - start_time, 3)
    add_log('extract_summary', source_name, execution_time_sec=execution_time, notes='Extract all Olist CSV files completed')
    return data

olist_data = extract_etl_source1()

customers = olist_data['customers']
geolocation = olist_data['geolocation']
order_items = olist_data['order_items']
payments = olist_data['payments']
reviews = olist_data['reviews']
orders = olist_data['orders']
products = olist_data['products']
sellers = olist_data['sellers']
translation = olist_data['translation']

print('Extract source 1 selesai.')

Extract source 1 selesai.


## 4. Extract Source 2 — Nager.Date Holiday API Brazil

In [ ]:
def extract_etl_source2():
    start_time = time.time()
    source_name = 'Nager.Date Public Holiday API'
    years = [2016, 2017, 2018]
    holiday_records = []

    for year in years:
        url = f'https://date.nager.at/api/v3/PublicHolidays/{year}/BR'
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        holiday_records.extend(data)

        add_log(
            step='extract',
            source_name=f'{source_name} - {year}',
            rows=len(data),
            cols=len(data[0].keys()) if len(data) > 0 else 0,
            notes=f'API endpoint: {url}'
        )

    holiday_df = pd.DataFrame(holiday_records)

    raw_holiday_file = f'{RAW_HOLIDAY_PATH}/brazil_holidays_2016_2018.csv'
    datalake_holiday_file = f'{DATALAKE_RAW_PATH}/brazil_holidays_2016_2018.csv'

    # Simpan raw API sebagai CSV. Kolom list akan otomatis tersimpan sebagai representasi string.
    holiday_df.to_csv(raw_holiday_file, index=False)
    holiday_df.to_csv(datalake_holiday_file, index=False)

    execution_time = round(time.time() - start_time, 3)
    add_log(
        step='extract_summary',
        source_name=source_name,
        rows=holiday_df.shape[0],
        cols=holiday_df.shape[1],
        file_size_mb=get_file_size_mb(raw_holiday_file),
        execution_time_sec=execution_time,
        notes='Holiday API data saved as raw CSV'
    )

    return holiday_df

holiday_raw = extract_etl_source2()
holiday_raw.head()

,date,localName,name,countryCode,fixed,global,counties,launchYear,types
0,2016-01-01,Confraternização Universal,New Year's Day,BR,False,True,None,None,[Public]
1,2016-02-08,Carnaval,Carnival,BR,False,True,None,None,"[Bank, Optional]"
2,2016-02-09,Carnaval,Carnival,BR,False,True,None,None,"[Bank, Optional]"
3,2016-03-25,Sexta-feira Santa,Good Friday,BR,False,True,None,None,[Public]
4,2016-03-27,Domingo de Páscoa,Easter Sunday,BR,False,True,None,None,[Public]


## 5. Data Understanding — Row, Column, Missing, Duplicate — FIXED

In [ ]:
datasets = {
    'customers': customers,
    'geolocation': geolocation,
    'order_items': order_items,
    'payments': payments,
    'reviews': reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'translation': translation,
    'holiday_raw': holiday_raw
}

# Kolom API seperti `types` berbentuk list.
# Pandas duplicated() tidak dapat memproses list karena list bersifat unhashable.
def safe_duplicate_count(df):
    temp_df = df.copy()
    for col in temp_df.columns:
        temp_df[col] = temp_df[col].apply(
            lambda x: json.dumps(x, sort_keys=True) if isinstance(x, (list, dict)) else x
        )
    return int(temp_df.duplicated().sum())

summary_rows = []
for name, df in datasets.items():
    summary_rows.append({
        'dataset': name,
        'rows': df.shape[0],
        'columns': df.shape[1],
        'missing_values_total': int(df.isnull().sum().sum()),
        'duplicate_rows': safe_duplicate_count(df)
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,dataset,rows,columns,missing_values_total,duplicate_rows
0,customers,99441,5,0,0
1,geolocation,1000163,5,0,261831
2,order_items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,145903,0
5,orders,99441,8,4908,0
6,products,32951,9,2448,0
7,sellers,3095,4,0,0
8,translation,71,2,0,0
9,holiday_raw,42,9,81,0


## 6. Standardisasi Nama Kolom

In [ ]:
def to_snake_case_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_', regex=False)
        .str.replace('-', '_', regex=False)
    )
    return df

customers = to_snake_case_columns(customers)
geolocation = to_snake_case_columns(geolocation)
order_items = to_snake_case_columns(order_items)
payments = to_snake_case_columns(payments)
reviews = to_snake_case_columns(reviews)
orders = to_snake_case_columns(orders)
products = to_snake_case_columns(products)
sellers = to_snake_case_columns(sellers)
translation = to_snake_case_columns(translation)
holiday_raw = to_snake_case_columns(holiday_raw)

print('Standardisasi nama kolom selesai.')

Standardisasi nama kolom selesai.


## 7. Transform — Convert Datetime

In [ ]:
start_transform = time.time()

order_date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in order_date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'], errors='coerce')

for col in ['review_creation_date', 'review_answer_timestamp']:
    if col in reviews.columns:
        reviews[col] = pd.to_datetime(reviews[col], errors='coerce')

holiday_df = holiday_raw.copy()
holiday_df['date'] = pd.to_datetime(holiday_df['date'], errors='coerce')

print('Datetime conversion selesai.')

Datetime conversion selesai.


## 8. Transform — Controlled Aggregation Sebelum Join

In [ ]:
payments_agg = payments.groupby('order_id', as_index=False).agg(
    payment_value=('payment_value', 'sum'),
    payment_installments=('payment_installments', 'sum'),
    payment_type=('payment_type', lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]),
    payment_count=('payment_sequential', 'count')
)

reviews_agg = reviews.groupby('order_id', as_index=False).agg(
    review_score=('review_score', 'mean'),
    review_count=('review_id', 'count')
)

products_en = products.merge(
    translation,
    on='product_category_name',
    how='left'
)

geolocation_agg = geolocation.groupby('geolocation_zip_code_prefix', as_index=False).agg(
    geolocation_lat=('geolocation_lat', 'mean'),
    geolocation_lng=('geolocation_lng', 'mean'),
    geolocation_city=('geolocation_city', 'first'),
    geolocation_state=('geolocation_state', 'first')
)

print('Aggregation sebelum join selesai.')
print('payments_agg:', payments_agg.shape)
print('reviews_agg:', reviews_agg.shape)
print('products_en:', products_en.shape)
print('geolocation_agg:', geolocation_agg.shape)

Aggregation sebelum join selesai.
payments_agg: (99440, 5)
reviews_agg: (98673, 3)
products_en: (32951, 10)
geolocation_agg: (19015, 5)


## 9. Transform — Controlled Join untuk Fact Table

In [ ]:
fact_df = order_items.copy()
fact_df['order_item_key'] = fact_df['order_id'].astype(str) + '_' + fact_df['order_item_id'].astype(str)

fact_df = fact_df.merge(orders, on='order_id', how='left')
fact_df = fact_df.merge(customers, on='customer_id', how='left')
fact_df = fact_df.merge(payments_agg, on='order_id', how='left')
fact_df = fact_df.merge(products_en, on='product_id', how='left')
fact_df = fact_df.merge(sellers, on='seller_id', how='left')
fact_df = fact_df.merge(reviews_agg, on='order_id', how='left')
fact_df = fact_df.merge(
    geolocation_agg,
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
)

print('Controlled join selesai.')
print('Fact shape:', fact_df.shape)
fact_df.head()

Controlled join selesai.
Fact shape: (112650, 42)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,order_item_key,customer_id,order_status,...,seller_zip_code_prefix,seller_city,seller_state,review_score,review_count,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,00010242fe8c5a6d1ba2dd792cb16214_1,3ce436f183e68e07877b285a838db11a,delivered,...,27277,volta redonda,SP,5.0,1.0,28013.0,-21.762775,-41.309633,campos dos goytacazes,RJ
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,00018f77f2f0320c557190d7a144bdd3_1,f6dd3ec061db4e3987629fe6b26e5cce,delivered,...,3471,sao paulo,SP,4.0,1.0,15775.0,-20.220527,-50.903424,santa fe do sul,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,000229ec398224ef6ca0657da4fc703e_1,6489ae5e4333f3693df5ad4372dab6d3,delivered,...,37564,borda da mata,MG,5.0,1.0,35661.0,-19.870305,-44.593326,pará de minas,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,00024acbcdf0a6daa1e931b038114c75_1,d4eb9395c8c0431ee92fce09860c5a06,delivered,...,14403,franca,SP,4.0,1.0,12952.0,-23.089925,-46.611654,atibaia,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,00042b26cf59d7ce69dfabb4e55b4fd9_1,58dbd0b2d70206bf40e62cd34e84d795,delivered,...,87900,loanda,PR,5.0,1.0,13226.0,-23.243402,-46.827614,varzea paulista,SP


## 10. Transform — Enrichment Holiday API

In [ ]:
holiday_clean = holiday_df[['date', 'localname', 'name', 'countrycode', 'global', 'types']].copy()
holiday_clean = holiday_clean.rename(columns={
    'date': 'holiday_date',
    'localname': 'holiday_local_name',
    'name': 'holiday_name',
    'countrycode': 'holiday_country_code',
    'types': 'holiday_types'
})

holiday_clean['holiday_date'] = pd.to_datetime(holiday_clean['holiday_date'], errors='coerce').dt.date
holiday_clean['holiday_types'] = holiday_clean['holiday_types'].apply(lambda x: ', '.join(x) if isinstance(x, list) else str(x))

fact_df['order_date'] = fact_df['order_purchase_timestamp'].dt.date

fact_df = fact_df.merge(
    holiday_clean,
    left_on='order_date',
    right_on='holiday_date',
    how='left'
)

fact_df['is_public_holiday'] = np.where(fact_df['holiday_name'].isna(), 0, 1)

print('Holiday enrichment selesai.')
fact_df[['order_date', 'holiday_name', 'is_public_holiday']].head()

Holiday enrichment selesai.


,order_date,holiday_name,is_public_holiday
0,2017-09-13,NaN,0
1,2017-04-26,NaN,0
2,2018-01-14,NaN,0
3,2018-08-08,NaN,0
4,2017-02-04,NaN,0


## 11. Transform — Feature Engineering

In [ ]:
fact_df['order_year'] = fact_df['order_purchase_timestamp'].dt.year
fact_df['order_month'] = fact_df['order_purchase_timestamp'].dt.month
fact_df['order_day'] = fact_df['order_purchase_timestamp'].dt.day
fact_df['order_day_name'] = fact_df['order_purchase_timestamp'].dt.day_name()
fact_df['order_hour'] = fact_df['order_purchase_timestamp'].dt.hour
fact_df['is_weekend'] = fact_df['order_day_name'].isin(['Saturday', 'Sunday']).astype(int)

fact_df['delivery_days'] = (
    fact_df['order_delivered_customer_date'] - fact_df['order_purchase_timestamp']
).dt.days

fact_df['estimated_delivery_days'] = (
    fact_df['order_estimated_delivery_date'] - fact_df['order_purchase_timestamp']
).dt.days

fact_df['delivery_delay_days'] = (
    fact_df['order_delivered_customer_date'] - fact_df['order_estimated_delivery_date']
).dt.days

fact_df['is_late_delivery'] = np.where(fact_df['delivery_delay_days'] > 0, 1, 0)
fact_df['total_item_value'] = fact_df['price'] + fact_df['freight_value']
fact_df['freight_ratio'] = np.where(fact_df['price'] > 0, fact_df['freight_value'] / fact_df['price'], np.nan)
fact_df['product_volume_cm3'] = fact_df['product_length_cm'] * fact_df['product_height_cm'] * fact_df['product_width_cm']
fact_df['delivery_status_group'] = np.where(fact_df['is_late_delivery'] == 1, 'late', 'on_time_or_early')

new_features = [
    'order_year', 'order_month', 'order_day_name', 'order_hour', 'is_weekend',
    'delivery_days', 'estimated_delivery_days', 'delivery_delay_days', 'is_late_delivery',
    'total_item_value', 'freight_ratio', 'product_volume_cm3', 'is_public_holiday'
]

fact_df[new_features].head()

,order_year,order_month,order_day_name,order_hour,is_weekend,delivery_days,estimated_delivery_days,delivery_delay_days,is_late_delivery,total_item_value,freight_ratio,product_volume_cm3,is_public_holiday
0,2017,9,Wednesday,8,0,7.0,15,-9.0,0,72.19,0.225637,3528.0,0
1,2017,4,Wednesday,10,0,16.0,18,-3.0,0,259.83,0.083076,60000.0,0
2,2018,1,Sunday,14,1,7.0,21,-14.0,0,216.87,0.089799,14157.0,0
3,2018,8,Wednesday,10,0,6.0,11,-6.0,0,25.78,0.984604,2400.0,0
4,2017,2,Saturday,13,1,25.0,40,-16.0,0,218.04,0.090745,42000.0,0


## 12. Transform — Handling Missing Value

In [ ]:
fact_clean = fact_df.copy()

numeric_fill_median = [
    'payment_value', 'payment_installments', 'review_score',
    'delivery_days', 'estimated_delivery_days', 'delivery_delay_days',
    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_volume_cm3',
    'freight_ratio'
]

for col in numeric_fill_median:
    if col in fact_clean.columns:
        median_value = fact_clean[col].median()
        fact_clean[col] = fact_clean[col].fillna(median_value)

categorical_fill_unknown = [
    'payment_type', 'product_category_name', 'product_category_name_english',
    'customer_city', 'customer_state', 'seller_city', 'seller_state',
    'holiday_name', 'holiday_local_name', 'holiday_country_code', 'holiday_types', 'delivery_status_group'
]

for col in categorical_fill_unknown:
    if col in fact_clean.columns:
        fact_clean[col] = fact_clean[col].fillna('Unknown')

fact_clean['is_public_holiday'] = fact_clean['is_public_holiday'].fillna(0).astype(int)

print('Missing value handling selesai.')
fact_clean.isnull().sum().sort_values(ascending=False).head(20)

Missing value handling selesai.


,0
holiday_date,109507
global,109507
order_delivered_customer_date,2454
product_name_lenght,1603
product_photos_qty,1603
product_description_lenght,1603
order_delivered_carrier_date,1194
review_count,942
geolocation_lat,302
geolocation_zip_code_prefix,302


## 13. Transform — Handling Duplicate dan Outlier

In [ ]:
before_rows = fact_clean.shape[0]
fact_clean = fact_clean.drop_duplicates(subset=['order_item_key'])
after_rows = fact_clean.shape[0]

print('Rows before duplicate handling:', before_rows)
print('Rows after duplicate handling:', after_rows)
print('Duplicate removed:', before_rows - after_rows)

outlier_cols = ['price', 'freight_value', 'payment_value', 'delivery_days', 'delivery_delay_days']
outlier_summary = []

for col in outlier_cols:
    if col in fact_clean.columns:
        q1 = fact_clean[col].quantile(0.25)
        q3 = fact_clean[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outlier_count = ((fact_clean[col] < lower) | (fact_clean[col] > upper)).sum()
        fact_clean[col + '_clipped'] = fact_clean[col].clip(lower, upper)
        outlier_summary.append({
            'column': col,
            'lower_bound': lower,
            'upper_bound': upper,
            'outlier_count': int(outlier_count)
        })

outlier_summary_df = pd.DataFrame(outlier_summary)
outlier_summary_df

Rows before duplicate handling: 112650
Rows after duplicate handling: 112650
Duplicate removed: 0


,column,lower_bound,upper_bound,outlier_count
0,price,-102.60000,277.40000,8427
1,freight_value,0.97500,33.25500,12134
2,payment_value,-128.89875,389.95125,9200
3,delivery_days,-7.50000,28.50000,5560
4,delivery_delay_days,-32.00000,8.00000,4926


## 14. Transform — Normalisasi Numerik dan Encoding Kategorikal

In [ ]:
normalize_cols = ['price_clipped', 'freight_value_clipped', 'payment_value_clipped']

for col in normalize_cols:
    min_val = fact_clean[col].min()
    max_val = fact_clean[col].max()
    if max_val != min_val:
        fact_clean[col + '_norm'] = (fact_clean[col] - min_val) / (max_val - min_val)
    else:
        fact_clean[col + '_norm'] = 0

encoding_cols = ['payment_type', 'customer_state', 'seller_state', 'product_category_name_english']

for col in encoding_cols:
    fact_clean[col + '_code'] = fact_clean[col].astype('category').cat.codes

fact_clean[[c + '_norm' for c in normalize_cols] + [c + '_code' for c in encoding_cols]].head()

,price_clipped_norm,freight_value_clipped_norm,payment_value_clipped_norm,payment_type_code,customer_state_code,seller_state_code,product_category_name_english_code
0,0.209908,0.381506,0.164580,2,18,22,21
1,0.864401,0.587206,0.657901,2,25,22,61
2,0.716507,0.523389,0.544956,2,10,8,40
3,0.043898,0.366016,0.042565,2,25,22,60
4,0.719761,0.531753,0.548032,2,25,15,43


## 15. Validasi Kualitas Data

In [ ]:
validation_results = []

def add_validation(rule_name, passed, details):
    validation_results.append({
        'rule_name': rule_name,
        'passed': bool(passed),
        'details': details
    })

add_validation(
    'uniqueness_check_order_item_key',
    fact_clean['order_item_key'].is_unique,
    f"duplicate order_item_key: {fact_clean['order_item_key'].duplicated().sum()}"
)

key_nulls = fact_clean[['order_id', 'order_item_id', 'product_id', 'seller_id']].isnull().sum().sum()
add_validation('null_check_primary_foreign_keys', key_nulls == 0, f'total null in key columns: {key_nulls}')

negative_price = (fact_clean['price'] < 0).sum()
negative_freight = (fact_clean['freight_value'] < 0).sum()
add_validation(
    'range_check_price_freight_non_negative',
    (negative_price == 0) and (negative_freight == 0),
    f'negative price: {negative_price}, negative freight: {negative_freight}'
)

add_validation(
    'datatype_consistency_order_purchase_timestamp',
    pd.api.types.is_datetime64_any_dtype(fact_clean['order_purchase_timestamp']),
    str(fact_clean['order_purchase_timestamp'].dtype)
)

product_ref_fail = (~fact_clean['product_id'].isin(products['product_id'])).sum()
add_validation('referential_integrity_product_id', product_ref_fail == 0, f'product_id not found in products: {product_ref_fail}')

review_out_of_range = ((fact_clean['review_score'] < 1) | (fact_clean['review_score'] > 5)).sum()
add_validation('distribution_range_review_score_1_to_5', review_out_of_range == 0, f'review_score out of range: {review_out_of_range}')

add_validation('row_count_minimum_100k', fact_clean.shape[0] >= 100000, f'final rows: {fact_clean.shape[0]}')
add_validation('column_count_minimum_12', fact_clean.shape[1] >= 12, f'final columns: {fact_clean.shape[1]}')

validation_df = pd.DataFrame(validation_results)
validation_df

,rule_name,passed,details
0,uniqueness_check_order_item_key,True,duplicate order_item_key: 0
1,null_check_primary_foreign_keys,True,total null in key columns: 0
2,range_check_price_freight_non_negative,True,"negative price: 0, negative freight: 0"
3,datatype_consistency_order_purchase_timestamp,True,datetime64[ns]
4,referential_integrity_product_id,True,product_id not found in products: 0
5,distribution_range_review_score_1_to_5,True,review_score out of range: 0
6,row_count_minimum_100k,True,final rows: 112650
7,column_count_minimum_12,True,final columns: 76


## 16. Simpan Data Curated ke Datalake

In [ ]:
curated_file = f'{DATALAKE_CURATED_PATH}/fact_order_items_curated.csv'
fact_clean.to_csv(curated_file, index=False)

validation_file = f'{DATALAKE_PROCESSED_PATH}/data_quality_validation_results.csv'
validation_df.to_csv(validation_file, index=False)

outlier_file = f'{DATALAKE_PROCESSED_PATH}/outlier_summary.csv'
outlier_summary_df.to_csv(outlier_file, index=False)

transform_time = round(time.time() - start_transform, 3)
add_log(
    step='transform_summary',
    source_name='Olist + Holiday API transformed curated fact',
    rows=fact_clean.shape[0],
    cols=fact_clean.shape[1],
    file_size_mb=get_file_size_mb(curated_file),
    execution_time_sec=transform_time,
    notes='Controlled join, cleaning, enrichment, feature engineering, validation completed'
)

print('Curated fact tersimpan di:', curated_file)
print('Validation results tersimpan di:', validation_file)
print('Outlier summary tersimpan di:', outlier_file)

Curated fact tersimpan di: /content/drive/MyDrive/bigdata_final_project/datalake/curated_zone/fact_order_items_curated.csv
Validation results tersimpan di: /content/drive/MyDrive/bigdata_final_project/datalake/processed_zone/data_quality_validation_results.csv
Outlier summary tersimpan di: /content/drive/MyDrive/bigdata_final_project/datalake/processed_zone/outlier_summary.csv


## 17. Desain Star Schema — Dimension Tables dan Fact Table

In [ ]:
dim_customer = fact_clean[[
    'customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state',
    'geolocation_lat', 'geolocation_lng'
]].drop_duplicates(subset=['customer_id'])

dim_product = fact_clean[[
    'product_id', 'product_category_name', 'product_category_name_english',
    'product_name_lenght', 'product_description_lenght', 'product_photos_qty',
    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_volume_cm3'
]].drop_duplicates(subset=['product_id'])

dim_seller = fact_clean[[
    'seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'
]].drop_duplicates(subset=['seller_id'])

dim_payment = fact_clean[[
    'payment_type', 'payment_installments', 'payment_count'
]].drop_duplicates().reset_index(drop=True)
dim_payment['payment_key'] = dim_payment.index + 1

order_dates = fact_clean[[
    'order_date', 'order_year', 'order_month', 'order_day', 'order_day_name',
    'is_weekend', 'is_public_holiday', 'holiday_name'
]].drop_duplicates()

dim_date = order_dates.copy()
dim_date['date_key'] = pd.to_datetime(dim_date['order_date']).dt.strftime('%Y%m%d').astype(int)
dim_date = dim_date.rename(columns={'order_date': 'date'})

fact_order_items = fact_clean.merge(
    dim_payment,
    on=['payment_type', 'payment_installments', 'payment_count'],
    how='left'
)

fact_order_items['date_key'] = pd.to_datetime(fact_order_items['order_date']).dt.strftime('%Y%m%d').astype(int)

fact_order_items = fact_order_items[[
    'order_item_key', 'order_id', 'order_item_id', 'customer_id', 'product_id', 'seller_id', 'payment_key', 'date_key',
    'price', 'freight_value', 'payment_value', 'review_score',
    'delivery_days', 'estimated_delivery_days', 'delivery_delay_days', 'is_late_delivery',
    'total_item_value', 'freight_ratio', 'is_public_holiday',
    'price_clipped', 'freight_value_clipped', 'payment_value_clipped',
    'price_clipped_norm', 'freight_value_clipped_norm', 'payment_value_clipped_norm',
    'payment_type_code', 'customer_state_code', 'seller_state_code', 'product_category_name_english_code'
]].drop_duplicates(subset=['order_item_key'])

print('dim_customer:', dim_customer.shape)
print('dim_product:', dim_product.shape)
print('dim_seller:', dim_seller.shape)
print('dim_payment:', dim_payment.shape)
print('dim_date:', dim_date.shape)
print('fact_order_items:', fact_order_items.shape)

dim_customer: (98666, 7)
dim_product: (32951, 11)
dim_seller: (3095, 4)
dim_payment: (94, 4)
dim_date: (616, 9)
fact_order_items: (112650, 29)


## 18. Buat File schema.sql untuk Dokumentasi PK/FK

In [ ]:
schema_sql = '''
-- Star Schema ETL Olist Data Warehouse
-- Grain fact table: 1 row = 1 item in 1 order

-- Primary Key Documentation:
-- dim_customer.customer_id
-- dim_product.product_id
-- dim_seller.seller_id
-- dim_payment.payment_key
-- dim_date.date_key
-- fact_order_items.order_item_key

-- Foreign Key Documentation:
-- fact_order_items.customer_id -> dim_customer.customer_id
-- fact_order_items.product_id -> dim_product.product_id
-- fact_order_items.seller_id -> dim_seller.seller_id
-- fact_order_items.payment_key -> dim_payment.payment_key
-- fact_order_items.date_key -> dim_date.date_key
'''

schema_path = f'{WAREHOUSE_PATH}/schema.sql'
with open(schema_path, 'w') as f:
    f.write(schema_sql)

print('schema.sql berhasil dibuat:', schema_path)

schema.sql berhasil dibuat: /content/drive/MyDrive/bigdata_final_project/warehouse/schema.sql


## 19. Load — Simpan Star Schema ke SQLite Data Warehouse

In [ ]:
start_load = time.time()

db_path = f'{WAREHOUSE_PATH}/bigdata_warehouse_etl.db'
conn = sqlite3.connect(db_path)

dim_customer.to_sql('dim_customer', conn, if_exists='replace', index=False)
dim_product.to_sql('dim_product', conn, if_exists='replace', index=False)
dim_seller.to_sql('dim_seller', conn, if_exists='replace', index=False)
dim_payment.to_sql('dim_payment', conn, if_exists='replace', index=False)
dim_date.to_sql('dim_date', conn, if_exists='replace', index=False)
fact_order_items.to_sql('fact_order_items', conn, if_exists='replace', index=False)

load_time = round(time.time() - start_load, 3)
add_log(
    step='load_summary',
    source_name='SQLite Data Warehouse - ETL Star Schema',
    rows=fact_order_items.shape[0],
    cols=fact_order_items.shape[1],
    file_size_mb=get_file_size_mb(db_path),
    execution_time_sec=load_time,
    notes=f'Data warehouse saved to {db_path}'
)

print('Load ke SQLite selesai:', db_path)

Load ke SQLite selesai: /content/drive/MyDrive/bigdata_final_project/warehouse/bigdata_warehouse_etl.db


## 20. Index dan Verifikasi Tabel Warehouse

In [ ]:
cursor = conn.cursor()

index_queries = [
    'CREATE INDEX IF NOT EXISTS idx_fact_order_id ON fact_order_items(order_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_customer_id ON fact_order_items(customer_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_product_id ON fact_order_items(product_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_seller_id ON fact_order_items(seller_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_date_key ON fact_order_items(date_key);'
]

for q in index_queries:
    cursor.execute(q)

conn.commit()

warehouse_tables = ['dim_customer', 'dim_product', 'dim_seller', 'dim_payment', 'dim_date', 'fact_order_items']
for table in warehouse_tables:
    count = pd.read_sql(f'SELECT COUNT(*) AS row_count FROM {table}', conn)
    print(table, count['row_count'].iloc[0])

dim_customer 98666
dim_product 32951
dim_seller 3095
dim_payment 94
dim_date 616
fact_order_items 112650


## 21. Query SQL Analitik

In [ ]:
analytic_queries = {}

analytic_queries['q1_monthly_revenue'] = """
SELECT
    d.order_year,
    d.order_month,
    COUNT(DISTINCT f.order_id) AS total_orders,
    COUNT(*) AS total_items,
    ROUND(SUM(f.total_item_value), 2) AS total_revenue
FROM fact_order_items f
JOIN dim_date d ON f.date_key = d.date_key
GROUP BY d.order_year, d.order_month
ORDER BY d.order_year, d.order_month;
"""

analytic_queries['q2_top_product_categories'] = """
SELECT
    p.product_category_name_english,
    COUNT(*) AS total_items,
    ROUND(SUM(f.total_item_value), 2) AS total_revenue,
    ROUND(AVG(f.review_score), 2) AS avg_review_score
FROM fact_order_items f
JOIN dim_product p ON f.product_id = p.product_id
GROUP BY p.product_category_name_english
ORDER BY total_revenue DESC
LIMIT 10;
"""

analytic_queries['q3_payment_type_distribution'] = """
SELECT
    dp.payment_type,
    COUNT(*) AS total_items,
    COUNT(DISTINCT f.order_id) AS total_orders,
    ROUND(SUM(f.payment_value), 2) AS total_payment_value
FROM fact_order_items f
JOIN dim_payment dp ON f.payment_key = dp.payment_key
GROUP BY dp.payment_type
ORDER BY total_payment_value DESC;
"""

analytic_queries['q4_customer_state_revenue'] = """
SELECT
    c.customer_state,
    COUNT(DISTINCT f.order_id) AS total_orders,
    COUNT(*) AS total_items,
    ROUND(SUM(f.total_item_value), 2) AS total_revenue
FROM fact_order_items f
JOIN dim_customer c ON f.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY total_revenue DESC
LIMIT 10;
"""

analytic_queries['q5_seller_state_revenue'] = """
SELECT
    s.seller_state,
    COUNT(DISTINCT f.seller_id) AS total_sellers,
    COUNT(*) AS total_items,
    ROUND(SUM(f.total_item_value), 2) AS total_revenue
FROM fact_order_items f
JOIN dim_seller s ON f.seller_id = s.seller_id
GROUP BY s.seller_state
ORDER BY total_revenue DESC
LIMIT 10;
"""

analytic_queries['q6_delivery_late_vs_review'] = """
SELECT
    f.is_late_delivery,
    COUNT(*) AS total_items,
    ROUND(AVG(f.delivery_delay_days), 2) AS avg_delivery_delay_days,
    ROUND(AVG(f.review_score), 2) AS avg_review_score
FROM fact_order_items f
GROUP BY f.is_late_delivery;
"""

analytic_queries['q7_holiday_vs_non_holiday'] = """
SELECT
    d.is_public_holiday,
    COUNT(DISTINCT f.order_id) AS total_orders,
    COUNT(*) AS total_items,
    ROUND(SUM(f.total_item_value), 2) AS total_revenue,
    ROUND(AVG(f.review_score), 2) AS avg_review_score
FROM fact_order_items f
JOIN dim_date d ON f.date_key = d.date_key
GROUP BY d.is_public_holiday;
"""

analytic_queries['q8_weekend_vs_weekday'] = """
SELECT
    d.is_weekend,
    COUNT(DISTINCT f.order_id) AS total_orders,
    COUNT(*) AS total_items,
    ROUND(SUM(f.total_item_value), 2) AS total_revenue,
    ROUND(AVG(f.review_score), 2) AS avg_review_score
FROM fact_order_items f
JOIN dim_date d ON f.date_key = d.date_key
GROUP BY d.is_weekend;
"""

# Simpan query SQL ke file analytic_queries.sql
queries_path = f"{WAREHOUSE_PATH}/analytic_queries.sql"

with open(queries_path, "w") as f:
    for name, query in analytic_queries.items():
        f.write("-- " + name + "\n")
        f.write(query.strip() + "\n\n")

print("8 query SQL analitik berhasil disimpan di:", queries_path)

# Jalankan dan tampilkan semua query
query_results = {}

for name, query in analytic_queries.items():
    print("\n==============================")
    print(name)
    print("==============================")
    result = pd.read_sql(query, conn)
    query_results[name] = result
    display(result.head(10))


8 query SQL analitik berhasil disimpan di: /content/drive/MyDrive/bigdata_final_project/warehouse/analytic_queries.sql

q1_monthly_revenue


,order_year,order_month,total_orders,total_items,total_revenue
0,2016,9,3,6,354.75
1,2016,10,308,363,56808.84
2,2016,12,1,1,19.62
3,2017,1,789,955,137188.49
4,2017,2,1733,1951,286280.62
5,2017,3,2641,3000,432048.59
6,2017,4,2391,2684,412422.24
7,2017,5,3660,4136,586190.95
8,2017,6,3217,3583,502963.04
9,2017,7,3969,4519,584971.62



q2_top_product_categories


,product_category_name_english,total_items,total_revenue,avg_review_score
0,health_beauty,9670,1441248.07,4.15
1,watches_gifts,5991,1305541.61,4.03
2,bed_bath_table,11115,1241681.72,3.91
3,sports_leisure,8641,1156656.48,4.11
4,computers_accessories,7827,1059272.40,3.94
5,furniture_decor,8334,902511.79,3.92
6,housewares,6964,778397.77,4.06
7,cool_stuff,3796,719329.95,4.15
8,auto,4235,685384.32,4.07
9,garden_tools,4347,584219.21,4.05



q3_payment_type_distribution


...

,is_late_delivery,total_items,avg_delivery_delay_days,avg_review_score
0,0,105385,-13.61,4.16
1,1,7265,10.49,2.33



q7_holiday_vs_non_holiday


,is_public_holiday,total_orders,total_items,total_revenue,avg_review_score
0,0,95878,109507,15415833.48,4.04
1,1,2788,3143,427719.76,3.97



q8_weekend_vs_weekday


,is_weekend,total_orders,total_items,total_revenue,avg_review_score
0,0,75965,87066,12232959.28,4.04
1,1,22701,25584,3610593.96,4.04


## 22. Simpan Log ETL dan Tutup Koneksi

In [ ]:
log_df = save_logs()
conn.close()
log_df

Log ETL tersimpan di: /content/drive/MyDrive/bigdata_final_project/etl_pipeline/logs/etl_process_log.csv


,timestamp,step,source_name,rows,cols,file_size_mb,execution_time_sec,status,notes
0,2026-05-20 07:54:27,extract,Kaggle Olist CSV Dataset - customers,99441.0,5.0,8.615,NaN,success,Raw file copied to /content/drive/MyDrive/bigd...
1,2026-05-20 07:54:40,extract,Kaggle Olist CSV Dataset - geolocation,1000163.0,5.0,58.435,NaN,success,Raw file copied to /content/drive/MyDrive/bigd...
2,2026-05-20 07:54:44,extract,Kaggle Olist CSV Dataset - order_items,112650.0,7.0,14.723,NaN,success,Raw file copied to /content/drive/MyDrive/bigd...
3,2026-05-20 07:54:47,extract,Kaggle Olist CSV Dataset - payments,103886.0,5.0,5.510,NaN,success,Raw file copied to /content/drive/MyDrive/bigd...
4,2026-05-20 07:54:52,extract,Kaggle Olist CSV Dataset - reviews,99224.0,7.0,13.782,NaN,success,Raw file copied to /content/drive/MyDrive/bigd...
5,2026-05-20 07:54:56,extract,Kaggle Olist CSV Dataset - orders,99441.0,8.0,16.837,NaN,success,Raw file copied to /content/drive/MyDrive/bigd...
6,2026-05-20 07:54:58,extract,Kaggle Olist CSV Dataset - products,32951.0,9.0,2.269,NaN,success,Raw file copied to /content/drive/MyDrive/bigd...
7,2026-05-20 07:54:59,extract,Kaggle Olist CSV Dataset - sellers,3095.0,4.0,0.167,NaN,success,Raw file copied to /content/drive/MyDrive/bigd...
8,2026-05-20 07:55:00,extract,Kaggle Olist CSV Dataset - translation,71.0,2.0,0.002,NaN,success,Raw file copied to /content/drive/MyDrive/bigd...
9,2026-05-20 07:55:00,extract_summary,Kaggle Olist CSV Dataset,NaN,NaN,NaN,37.146,success,Extract all Olist CSV files completed
